## 1. Install Dependencies

In [ ]:
!pip install ultralytics pandas numpy matplotlib psutil onnxruntime tensorflow -q

In [ ]:
# Import libraries
import os
import gc
import time
import json
from datetime import datetime

import torch
import psutil
import pandas as pd
import numpy as np
import tensorflow as tf
import onnxruntime as ort
import matplotlib.pyplot as plt
from ultralytics import YOLO

## 2. Configuration

In [ ]:
# ==> UPDATE THESE PATHS TO YOUR UPLOADED MODELS <==
MODEL_PATHS = {
    'pytorch': '/content/yolov8n_pytorch_float16.pt',
    'onnx': '/content/yolov8n_best_int8_dynamic.onnx',
    'tflite': '/content/yolov8n_float16.tflite'
}

# Benchmark settings
BENCHMARK_CONFIG = {
    "img_size": 640,
    "num_runs": 100,
    "warmup_runs": 10,
    "batch_sizes": [1, 4, 8, 16]
}

print("📋 Configuration set!")
print(f"Looking for models at:")
for fmt, path in MODEL_PATHS.items():
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"  {exists} {fmt}: {path}")

## 3. Core Benchmarking Functions

In [ ]:
def benchmark_model(model_path, format_type, img_size=640, num_runs=100, warmup_runs=10, batch_size=1):
    """Run performance benchmark for a model"""

    # Clean memory
    gc.collect()
    if format_type == 'pytorch':
        torch.cuda.empty_cache()

    start_mem = psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

    # Setup model and inference function
    if format_type == 'pytorch':
        model = YOLO(model_path)
        test_images = [np.ones((img_size, img_size, 3), dtype=np.uint8) * 128 for _ in range(batch_size)]
        def run_inference():
            return model(test_images, verbose=False)

    elif format_type == 'onnx':
        session = ort.InferenceSession(model_path, providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        input_name = session.get_inputs()[0].name
        test_input = np.random.rand(batch_size, 3, img_size, img_size).astype(np.float32)
        def run_inference():
            return session.run(None, {input_name: test_input})

    elif format_type == 'tflite':
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        input_details = interpreter.get_input_details()[0]

        # TFLite doesn't play nice with batching
        if batch_size > 1:
            print(f"TFLite: forcing batch_size=1")
            batch_size = 1

        test_input = np.random.randint(0, 256, (batch_size, img_size, img_size, 3), dtype=np.uint8)
        if input_details['dtype'] == np.float32:
            test_input = test_input.astype(np.float32)

        def run_inference():
            interpreter.set_tensor(input_details['index'], test_input)
            interpreter.invoke()
            output_details = interpreter.get_output_details()
            return [interpreter.get_tensor(o['index']) for o in output_details]

    # Warmup
    print(f"🔥 Warming up {format_type} (batch={batch_size})...")
    for _ in range(warmup_runs):
        _ = run_inference()

    # Benchmark
    print(f"⚡ Benchmarking {format_type}...")
    latencies = []
    for _ in range(num_runs):
        start = time.time()
        _ = run_inference()
        latencies.append((time.time() - start) * 1000)  # ms

    # Calculate metrics
    end_mem = psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024
    avg_latency = np.mean(latencies)
    throughput = (1000 / avg_latency) * batch_size

    results = {
        "format": format_type,
        "batch_size": batch_size,
        "avg_latency_ms": round(avg_latency, 2),
        "p95_latency_ms": round(np.percentile(latencies, 95), 2),
        "throughput_fps": round(throughput, 2),
        "memory_usage_mb": round(end_mem - start_mem, 2),
        "image_size": img_size
    }

    print(f"✅ {format_type}: {throughput:.1f} FPS, {avg_latency:.1f}ms")
    return results

print("🛠️ Benchmark function ready!")

## 4. Visualization Function

In [ ]:
def plot_results(results_df, output_dir):
    """Generate comparison plots"""
    fig, axs = plt.subplots(2, 2, figsize=(15, 10))
    colors = {'pytorch': 'blue', 'onnx': 'orange', 'tflite': 'green'}

    # Batch size 1 comparisons
    batch1 = results_df[results_df['batch_size'] == 1].set_index('format')

    # Latency
    batch1['avg_latency_ms'].plot(kind='bar', ax=axs[0,0],
                                  color=[colors.get(i, 'gray') for i in batch1.index])
    axs[0,0].set_title('Latency (ms) - Lower is Better')
    axs[0,0].grid(axis='y', alpha=0.3)

    # Throughput
    batch1['throughput_fps'].plot(kind='bar', ax=axs[0,1],
                                  color=[colors.get(i, 'gray') for i in batch1.index])
    axs[0,1].set_title('Throughput (FPS) - Higher is Better')
    axs[0,1].grid(axis='y', alpha=0.3)

    # Memory
    batch1['memory_usage_mb'].plot(kind='bar', ax=axs[1,0],
                                   color=[colors.get(i, 'gray') for i in batch1.index])
    axs[1,0].set_title('Memory Usage (MB)')
    axs[1,0].grid(axis='y', alpha=0.3)

    # Batch scaling
    for fmt, color in colors.items():
        fmt_data = results_df[results_df['format'] == fmt]
        if len(fmt_data) > 0:
            axs[1,1].plot(fmt_data['batch_size'], fmt_data['throughput_fps'],
                         'o-', label=fmt, color=color)
    axs[1,1].set_title('Batch Size vs Throughput')
    axs[1,1].set_xlabel('Batch Size')
    axs[1,1].set_ylabel('FPS')
    axs[1,1].legend()
    axs[1,1].grid(alpha=0.3)

    plt.tight_layout()
    plot_path = f"{output_dir}/benchmark_results.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    return plot_path

print("📊 Plotting function ready!")

## 5. Main Benchmark Runner

In [ ]:
def run_benchmark():
    """Main function - keep it simple"""

    # Setup output directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"/content/benchmark_results_{timestamp}"
    os.makedirs(output_dir, exist_ok=True)
    print(f"📁 Results will be saved to: {output_dir}")

    # Find available models
    available_models = {fmt: path for fmt, path in MODEL_PATHS.items()
                       if os.path.exists(path)}

    if not available_models:
        print("❌ No models found! Check your MODEL_PATHS")
        print("Expected files:")
        for fmt, path in MODEL_PATHS.items():
            print(f"  {fmt}: {path}")
        return

    print(f"📊 Found models: {list(available_models.keys())}")

    # Run benchmarks
    all_results = []
    for format_type, model_path in available_models.items():

        # TFLite only gets batch_size=1
        batch_sizes = [1] if format_type == 'tflite' else BENCHMARK_CONFIG['batch_sizes']

        for batch_size in batch_sizes:
            try:
                result = benchmark_model(
                    model_path, format_type,
                    img_size=BENCHMARK_CONFIG['img_size'],
                    num_runs=BENCHMARK_CONFIG['num_runs'],
                    batch_size=batch_size
                )
                all_results.append(result)
            except Exception as e:
                print(f"❌ Failed {format_type} batch={batch_size}: {e}")

    if not all_results:
        print("No benchmarks completed successfully")
        return

    # Save and display results
    df = pd.DataFrame(all_results)

    # Save files
    csv_path = f"{output_dir}/results.csv"
    json_path = f"{output_dir}/results.json"
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient='records', indent=2)

    # Display summary
    print(f"\n🎯 BENCHMARK RESULTS")
    print("="*50)
    print(df.to_string(index=False))

    # Generate plots
    plot_path = plot_results(df, output_dir)

    print(f"\n💾 Results saved to:")
    print(f"  📊 {csv_path}")
    print(f"  📈 {plot_path}")

    return df

print("🚀 Main function ready!")

## 6. Run the Benchmark!

In [ ]:
# Execute the benchmark
results = run_benchmark()

## **ANALYSIS SUMMARY**

This analysis presents a comparative evaluation of three deep learning model formats—PyTorch, ONNX (Open Neural Network Exchange), and TensorFlow Lite (TFLite)—based on empirical performance benchmarks. The objective is to ascertain the optimal format for production deployment by examining key performance indicators: inference latency, throughput rate, memory consumption, and performance scalability with increasing batch size. The findings suggest that while TFLite offers the lowest single-instance latency, the ONNX format provides a superior balance of computational speed and memory efficiency, rendering it the most suitable candidate for deployment.

---

## 1. Comparative Performance Analysis

An evaluation of the provided benchmark data reveals significant performance disparities among the three model formats. The analysis is structured around three primary metrics: computational performance (latency and throughput), memory resource allocation, and scalability.

#### **1.1. Computational Performance: Latency and Throughput**

For single-batch inference tasks ($n=1$), the TFLite model demonstrated the highest level of computational performance. It exhibited the lowest mean inference latency at $207.22$ ms, which corresponds to the highest throughput of $4.83$ frames per second (FPS). The ONNX model presented a comparable performance profile, with a mean latency of $225.23$ ms and a throughput of $4.44$ FPS. The native PyTorch model was found to be the least performant in this configuration, registering the highest mean latency ($262.82$ ms) and the lowest throughput ($3.80$ FPS).

#### **1.2. Memory Footprint Analysis**

The most substantial variance between the formats was observed in memory resource consumption. The baseline PyTorch model exhibited a considerable memory footprint of $180.47$ MB. In stark contrast, the optimized formats designed for inference demonstrated profound efficiency. The ONNX model consumed a minimal $4.05$ MB of memory, representing a **97.7% reduction** relative to the PyTorch model. The TFLite model was also highly efficient, utilizing $8.54$ MB. These results underscore the critical impact of model conversion and optimization on resource allocation in a production environment.

#### **1.3. Scalability with Batch Processing**

The investigation into performance scalability was limited to the PyTorch model. The results indicate that throughput improves non-linearly, increasing from $3.80$ FPS at a batch size of 1 to a peak of $4.22$ FPS at a batch size of 8. However, a further increase in batch size to 16 resulted in a performance degradation, with the throughput rate declining to $3.61$ FPS. This phenomenon suggests the onset of a computational bottleneck or memory bandwidth saturation beyond a batch size of 8 for this specific model and hardware configuration.

---

## 2. Conclusion and Recommendation

The empirical results lead to a clear and definitive recommendation regarding the choice of model format for production deployment.

The native PyTorch model, while essential for the training and development phase, is demonstrably ill-suited for production inference due to its substantial latency and excessive memory footprint.

The choice between TFLite and ONNX is more nuanced. While the TFLite model holds a marginal advantage in single-instance inference speed, the **ONNX model presents the most compelling and balanced performance profile.** Its combination of high throughput and exceptionally low memory consumption makes it a superior choice for a wide array of deployment scenarios.

Therefore, it is formally recommended that the **ONNX format be adopted for production deployment.** This format ensures high computational efficiency and minimal resource consumption, making it optimally suited for both resource-constrained edge devices and cost-sensitive, scalable cloud infrastructures.

## 7. Optional: Quick Results Summary

In [ ]:
# Display a quick summary if you want
if 'results' in locals() and results is not None:
    print("\n📈 QUICK SUMMARY")
    print("-" * 30)

    # Best performers
    fastest = results.loc[results['throughput_fps'].idxmax()]
    lowest_latency = results.loc[results['avg_latency_ms'].idxmin()]

    print(f"🏆 Highest Throughput: {fastest['format']} - {fastest['throughput_fps']} FPS")
    print(f"⚡ Lowest Latency: {lowest_latency['format']} - {lowest_latency['avg_latency_ms']} ms")

    # Format comparison (batch size 1)
    batch1 = results[results['batch_size'] == 1]
    if len(batch1) > 0:
        print(f"\n📊 Format Comparison (Batch Size 1):")
        for _, row in batch1.iterrows():
            print(f"  {row['format']:8}: {row['throughput_fps']:6.1f} FPS, {row['avg_latency_ms']:6.1f}ms")

## Usage Instructions:

1. **Upload your models** to `/content/models/` or update the paths in step 2
2. **Run each cell** in order  
3. **Check results** in the generated plots and CSV files
4. **Download results** from the `/content/benchmark_results_[timestamp]/` folder